# 10. Submission Packaging and Provenance

- Parent issue: `#20`
- Status: `active`
- Summary: Point to the implemented packaging entrypoint + provenance manifest policy for Kaggle submissions.


## Audience and Why It Matters

Submission owners, reviewers, and project leads managing leaderboard attempts.

## Decision / Hypothesis

Keep `submission.zip` minimal (two adapter files at the zip root) and keep provenance out-of-band as `submission.manifest.json` beside the zip so the archive stays Kaggle-clean while remaining auditable.

## Environment and Reproduction

This notebook is a thin runbook that points at the implemented packaging code.

- Python: 3.11+
- Run from repo root (or open directly in Jupyter)
- Implementation: `src/inference/submission.py`
- CLI: `scripts/package_submission.py`
- Tests: `tests/inference/test_submission_packaging.py`
- Recommended output dir: `experiments/submissions/<run_id>/` (keep artifacts out of git)
- Registry entry: `docs/execution/NOTEBOOKS.md`


In [ ]:
from pathlib import Path
import platform

REPO_ROOT = Path.cwd()

print(f"Repo root: {REPO_ROOT}")
print(f"Python platform: {platform.platform()}")


## Method and Outputs

### Canonical packaging contract
- `submission.zip` MUST contain EXACTLY: `adapter_config.json` + `adapter_model.safetensors` at the zip root.
- `submission.manifest.json` MUST be written beside the zip (never inside).

### Expected outputs
- `experiments/submissions/<run_id>/submission.zip`
- `experiments/submissions/<run_id>/submission.manifest.json`

### Example command
```bash
python scripts/package_submission.py \
  --adapter-dir adapters/<your_adapter_dir> \
  --output-dir experiments/submissions/<run_id> \
  --base-model-id metric/nemotron-3-nano-30b-a3b-bf16/transformers/default \
  --adapter-rank 32 \
  --dataset-version <dataset_version> \
  --eval-sha <eval_sha>
```


In [ ]:
tasks = [
    "Run packaging tests: pytest tests/inference/test_submission_packaging.py -q",
    "Package a real adapter dir into experiments/submissions/<run_id>/",
    "Verify zip layout: python -m zipfile -l experiments/submissions/<run_id>/submission.zip",
]

for item in tasks:
    print(f"- {item}")


## Results / Open Risks

- Packaging code + tests exist; this notebook is now a runbook scaffold.
- Remaining risk: submitting the wrong adapter bytes (path mix-ups). Mitigate by always checking the out-of-band manifest + SHA256 hashes before uploading.
- Future work: add optional dry-run eval gates once real `#18/#19` artifacts exist.


## Sources

- [Architecture doc](docs/architecture/ARCHITECTURE.md)
- [Competition facts](docs/architecture/COMPETITION.md)
- [Kaggle submission demo](https://www.kaggle.com/code/ryanholbrook/nvidia-nemotron-submission-demo)
- [Packager implementation](src/inference/submission.py)
- [Packager CLI](scripts/package_submission.py)
- [Packager tests](tests/inference/test_submission_packaging.py)
